In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import math
import matplotlib.pyplot as plt
import pandas as pd

# ===== Setting ===== #
obs_size = 500
theta_dim = 3
x_dim = 5

a1 = 0.0
b1 = 10.0

a2 = 0.0
b2 = 10.0

a3 = 0.01 #
b3 = 0.5

In [ ]:
theta_true = torch.tensor([1.0, 5.0, 0.2], dtype = torch.float32)

## Table

In [ ]:
# ======= single model ======= #
error_1model = torch.zeros(100, 3)
width_1model = torch.zeros(100, 3)
cover_1model = torch.zeros(100, 3)

for obs_id in range(100):
    samples = torch.from_numpy(np.load(f"res_single_model/samples_obs{obs_id}.npy"))
    theta_post = samples[:, 1000::5, :].reshape(-1, 3)
    theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]

    error_1model[obs_id] = (theta_post.mean(dim = 0) - theta_true).abs()
    
    CI_lower = torch.quantile(theta_post, 0.025, dim = 0)
    CI_upper = torch.quantile(theta_post, 0.975, dim = 0)
    width_1model[obs_id] = CI_upper - CI_lower
    cover_1model[obs_id] = ((theta_true >= CI_lower) & (theta_true <= CI_upper)).float()


In [ ]:
# ======= n-model ======= #
error_nmodel = torch.zeros(100, 3)
width_nmodel = torch.zeros(100, 3)
cover_nmodel = torch.zeros(100, 3)

for obs_id in range(100):
    samples = torch.from_numpy(np.load(f"res_nmodel/samples_obs{obs_id}.npy"))
    theta_post = samples[:, 1000::5, :].reshape(-1, 3)
    theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]

    error_nmodel[obs_id] = (theta_post.mean(dim = 0) - theta_true).abs()
    CI_lower = torch.quantile(theta_post, 0.025, dim = 0)
    CI_upper = torch.quantile(theta_post, 0.975, dim = 0)
    width_nmodel[obs_id] = CI_upper - CI_lower
    cover_nmodel[obs_id] = ((theta_true >= CI_lower) & (theta_true <= CI_upper)).float()

In [ ]:
# ======= ABC ======= #
error_ABC = torch.zeros(100, 3) 
width_ABC = torch.zeros(100, 3)
cover_ABC = torch.zeros(100, 3)

for obs_id in range(100):
    ABCW1 = pd.read_csv(f"res/ABCW1_x_obs_{obs_id}.csv")
    ABCW1 = torch.tensor(ABCW1.values, dtype = torch.float32).contiguous()
    ABCW1[:, 1] = ABCW1[:, 0] + ABCW1[:, 1]

    error_ABC[obs_id] = (ABCW1.mean(dim = 0) - theta_true).abs()

    CIr1_lower = torch.quantile(ABCW1, 0.025, dim = 0)
    CIr1_upper = torch.quantile(ABCW1, 0.975, dim = 0)
    width_ABC[obs_id] = CIr1_upper - CIr1_lower
    cover_ABC[obs_id] = ((theta_true >= CIr1_lower) & (theta_true <= CIr1_upper)).float()

In [ ]:
# ======= BSL ======= #
error_BSL = torch.zeros(100, 3) 
width_BSL = torch.zeros(100, 3)
cover_BSL = torch.zeros(100, 3)

for obs_id in range(100):
    BSL = pd.read_csv(f"res/BSL_x_obs_{obs_id}.csv")
    BSL = torch.tensor(BSL.values, dtype = torch.float32).contiguous()
    BSL[:, 1] = BSL[:, 0] + BSL[:, 1] 
    CIr1_lower = torch.quantile(BSL, 0.025, dim = 0)
    CIr1_upper = torch.quantile(BSL, 0.975, dim = 0)
    
    error_BSL[obs_id] = ( BSL.mean(dim = 0) - theta_true ).abs() 
    width_BSL[obs_id] = CIr1_upper - CIr1_lower
    cover_BSL[obs_id] = torch.logical_and(theta_true < CIr1_upper, theta_true > CIr1_lower)

In [ ]:
# ======= NPE ======= #
error_NPE = torch.zeros(100, 3)
width_NPE = torch.zeros(100, 3)
cover_NPE = torch.zeros(100, 3)

for obs_id in range(100):
    theta_post = torch.from_numpy(np.load(f"NPE_res/samples_obs{obs_id}.npy"))
    # theta_post = torch.from_numpy(np.load(f"NPE_res_newdeepsets/samples_obs{obs_id}.npy"))
    theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]

    error_NPE[obs_id] = (theta_post.mean(dim = 0) - theta_true).abs()
    CI_lower = torch.quantile(theta_post, 0.025, dim = 0)
    CI_upper = torch.quantile(theta_post, 0.975, dim = 0)
    width_NPE[obs_id] = CI_upper - CI_lower
    cover_NPE[obs_id] = ((theta_true >= CI_lower) & (theta_true <= CI_upper)).float()

In [ ]:
# ======= NLE ======= #
error_NLE = torch.zeros(100, 3)
width_NLE = torch.zeros(100, 3)
cover_NLE = torch.zeros(100, 3)

for obs_id in range(100):
    theta_post = torch.from_numpy(np.load(f"res_inference/NLE/samples{obs_id}.npy"))
    theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]

    error_NLE[obs_id] = (theta_post.mean(dim = 0) - theta_true).abs()
    CI_lower = torch.quantile(theta_post, 0.025, dim = 0)
    CI_upper = torch.quantile(theta_post, 0.975, dim = 0)
    width_NLE[obs_id] = CI_upper - CI_lower
    cover_NLE[obs_id] = ((theta_true >= CI_lower) & (theta_true <= CI_upper)).float()

In [ ]:
num_rounds = 5
budget = 20000 

error_SNPE = torch.zeros(100, 3)
width_SNPE = torch.zeros(100, 3)
cover_SNPE = torch.zeros(100, 3)

theta_true_aug = torch.tensor([1.0, 5.0, 0.2])
for task_id in range(100):
    round_id = num_rounds - 1
    # load if it exists
    if os.path.exists(f"res_NPE_embed_sequential_{num_rounds}rounds_{budget}budget/samples_task{task_id}_round{round_id}.npy"):
        theta_post = torch.from_numpy(np.load(f"res_NPE_embed_sequential_{num_rounds}rounds_{budget}budget/samples_task{task_id}_round{round_id}.npy"))
        theta_post_aug = theta_post.clone()
        theta_post_aug[:, 1] = theta_post_aug[:, 0] + theta_post[:, 1]
        error_SNPE[task_id] = (theta_post_aug.mean(dim = 0) - theta_true_aug).abs()
        CI_upper = theta_post_aug.quantile(0.975, dim=0)
        CI_lower = theta_post_aug.quantile(0.025, dim=0)
        width_SNPE[task_id] = CI_upper - CI_lower
        cover_SNPE[task_id] = ((theta_true_aug >= CI_lower) & (theta_true_aug <= CI_upper)).float()


In [ ]:
import pandas as pd

methods = [
    ("single-model", error_1model, width_1model, cover_1model),
    ("n-model",      error_nmodel, width_nmodel, cover_nmodel),
    ("ABC",          error_ABC,    width_ABC,    cover_ABC),
    ("BSL",          error_BSL,    width_BSL,    cover_BSL),
    ("NPE",          error_NPE,    width_NPE,    cover_NPE),
    ("NLE",          error_NLE,    width_NLE,    cover_NLE),
    ("SNPE",         error_SNPE,    width_SNPE,    cover_SNPE),
]

params = ["theta_1", "theta_2", "theta_3"]
metrics = ["error", "Width", "Cover"]

columns = pd.MultiIndex.from_tuples([
    ("theta_1", "error"),
    ("theta_1", "Width"),
    ("theta_1", "Cover"),
    ("theta_2", "error"),
    ("theta_2", "Width"),
    ("theta_2", "Cover"),
    ("theta_3", r"error ($\times 10^{-2}$)"),
    ("theta_3", "Width"),
    ("theta_3", "Cover"),
])

table_df = pd.DataFrame(
    index=[m[0] for m in methods],
    columns=columns
)

def format_mean_std(mean_val, std_val, digits=3):
    return (
        f"\\begin{{tabular}}{{@{{}}c@{{}}}} "
        f"{mean_val:.{digits}f} \\\\ ({std_val:.{digits}f}) "
        f"\\end{{tabular}}"
    )

for method_name, err, wid, cov in methods:
    err_mean = err.mean(dim=0)
    err_std  = err.std(dim=0)

    wid_mean = wid.mean(dim=0)
    wid_std  = wid.std(dim=0)

    cov_mean = cov.float().mean(dim=0)

    # theta_1
    table_df.loc[method_name, ("theta_1", "error")] = format_mean_std(
        err_mean[0].item(), err_std[0].item(), digits=3
    )
    table_df.loc[method_name, ("theta_1", "Width")] = format_mean_std(
        wid_mean[0].item(), wid_std[0].item(), digits=3
    )
    table_df.loc[method_name, ("theta_1", "Cover")] = f"{cov_mean[0].item():.2f}"

    # theta_2
    table_df.loc[method_name, ("theta_2", "error")] = format_mean_std(
        err_mean[1].item(), err_std[1].item(), digits=3
    )
    table_df.loc[method_name, ("theta_2", "Width")] = format_mean_std(
        wid_mean[1].item(), wid_std[1].item(), digits=3
    )
    table_df.loc[method_name, ("theta_2", "Cover")] = f"{cov_mean[1].item():.2f}"

    # theta_3: error scaled by 1e-2
    table_df.loc[method_name, ("theta_3", r"error ($\times 10^{-2}$)")] = format_mean_std(
        err_mean[2].item() * 100, err_std[2].item() * 100, digits=3
    )
    table_df.loc[method_name, ("theta_3", "Width")] = format_mean_std(
        wid_mean[2].item(), wid_std[2].item(), digits=3
    )
    table_df.loc[method_name, ("theta_3", "Cover")] = f"{cov_mean[2].item():.2f}"

table_df


In [ ]:
latex_code = table_df.to_latex(
    escape=False,
    multicolumn=True,
    multirow=True
)

print(latex_code)


## Plot

In [ ]:
obs_id = 0

# single model
samples = torch.from_numpy(np.load(f"res_single_model/samples_obs{obs_id}.npy"))
theta_post = samples[:, 1000::5, :].reshape(-1, 3)
theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]
res_single = theta_post.clone()

# n-model
samples = torch.from_numpy(np.load(f"res_nmodel/samples_obs{obs_id}.npy"))
theta_post = samples[:, 1000::5, :].reshape(-1, 3)
theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]
res_n = theta_post.clone()

# ABC
ABCW1 = pd.read_csv(f"res/ABCW1_x_obs_{obs_id}.csv")
ABCW1 = torch.tensor(ABCW1.values, dtype = torch.float32).contiguous()
ABCW1[:, 1] = ABCW1[:, 0] + ABCW1[:, 1]
res_ABC = ABCW1.clone()

# BSL
BSL = pd.read_csv(f"res/BSL_x_obs_{obs_id}.csv")
BSL = torch.tensor(BSL.values, dtype = torch.float32).contiguous()
BSL[:, 1] = BSL[:, 0] + BSL[:, 1] 
res_BSL = BSL.clone()

# NPE
theta_post = torch.from_numpy(np.load(f"NPE_res/samples_obs{obs_id}.npy"))
theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]
res_NPE = theta_post.clone()

# NLE
theta_post = torch.from_numpy(np.load(f"res_inference/NLE/samples{obs_id}.npy"))
theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]
res_NLE = theta_post.clone()

# SNPE
theta_post = torch.from_numpy(np.load(f"res_NPE_embed_sequential_{5}rounds_{20000}budget/samples_task{obs_id}_round{4}.npy"))
theta_post[:, 1] = theta_post[:, 0] + theta_post[:, 1]
res_SNPE = theta_post.clone()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ours_np = res_n.numpy()
ours_1_np = res_single.numpy()
abcw1_np = res_ABC.numpy()
bsl_np = res_BSL.numpy()
npe_np = res_NPE.numpy()
nle_np = res_NLE.numpy()
snpe_np = res_SNPE.numpy()

methods = ["n-model", "single-model", "ABC", "BSL", "NPE", "NLE", "SNPE"]
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown", "tab:pink"]
data_list = [ours_np, ours_1_np, abcw1_np, bsl_np, npe_np, nle_np, snpe_np]
bw_list = [2.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

# Manually set x-limits for each subplot
xlim_list = [
    (0.97, 1.008),   # first subplot: theta_1, n-model only
    (0, 3.5),       # second subplot: theta_1
    (2, 8),       # theta_2
    (0.15, 0.28),    # theta_3
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i in range(4):
    ax = axes[i]

    if i == 0:
        active_js = [0]
        dim = 0
        title = r"$\theta_{1}$"
    else:
        dim = i - 1
        title = rf"$\theta_{{{i}}}$"

        if i == 1:
            active_js = [j for j in range(len(methods)) if j not in [0]]
        else:
            active_js = list(range(len(methods)))

    for j in active_js:
        sns.kdeplot(
            data_list[j][:, dim],
            ax=ax,
            fill=True,
            alpha=0.10,
            bw_adjust=bw_list[j],
            label=methods[j],
            color=colors[j],
            # cut=0
        )

    ax.axvline(
        x=theta_true[dim],
        color="black",
        linestyle="--",
        linewidth=1,
        label="Truth" if i == 3 else None,
        zorder=10
    )

    ax.set_xlim(xlim_list[i])
    ax.set_title(title, fontsize=20)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)

    if i == 0:
        ax.set_ylabel("Density", fontsize=20)
    else:
        ax.set_ylabel("")

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    fontsize=20,
    ncol=len(labels)
)

plt.tight_layout(rect=[0, 0.18, 1, 1])
# plt.savefig("Figs/queuing_represent.pdf", dpi=300, bbox_inches="tight")
plt.show()